In [1]:
data = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes'],
]

def more_general(h1, h2):
    return all(
        h1[i] == '?' or (h1[i] != 'phi' and h1[i] == h2[i])
        for i in range(len(h1))
    )

def candidate_elimination(data):
    n = len(data[0]) - 1

    S = ['phi'] * n
    G = [['?'] * n]

    for example in data:
        x = example[:-1]
        label = example[-1]

        if label == 'Yes':
            # Remove inconsistent hypotheses from G
            G = [g for g in G if more_general(g, x)]

            # Update S
            for i in range(n):
                if S[i] == 'phi':
                    S[i] = x[i]
                elif S[i] != x[i]:
                    S[i] = '?'

        else:  # Negative example
            new_G = []
            for g in G:
                if more_general(g, x):
                    for i in range(n):
                        if g[i] == '?':
                            if x[i] != S[i]:
                                new_h = g.copy()
                                new_h[i] = S[i]
                                new_G.append(new_h)
                        elif g[i] != x[i]:
                            new_G.append(g)
                else:
                    new_G.append(g)
            G = new_G

    return [S], G


S, G = candidate_elimination(data)

print("S (Specific boundary):", S)
print("G (General boundary):", G)

S (Specific boundary): [['Sunny', 'Warm', '?', 'Strong', '?', '?']]
G (General boundary): [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?']]


In [1]:
import numpy as np

def learn(concepts, target):
    # Initialize Specific boundary with the first positive example
    specific_h = concepts[0].copy()
    
    # Initialize General boundary with '?' (most general)
    general_h = [["?" for _ in range(len(specific_h))] for _ in range(len(specific_h))]

    for i, h in enumerate(concepts):
        if target[i] == "Yes":
            # If positive example, update Specific Boundary
            for x in range(len(specific_h)):
                if h[x] != specific_h[x]:
                    specific_h[x] = '?'
                    general_h[x][x] = '?'
        
        else:
            # If negative example, update General Boundary
            for x in range(len(specific_h)):
                if h[x] != specific_h[x] and specific_h[x] != '?':
                    general_h[x][x] = specific_h[x]
                else:
                    general_h[x][x] = '?'

    # Filter out empty or redundant hypotheses from general_h
    indices = [i for i, val in enumerate(general_h) if val == ['?'] * len(specific_h)]
    for i in reversed(indices):
        general_h.pop(i)
        
    return specific_h, general_h

# --- Example Usage ---
# Dataset: Sky, Temp, Humid, Wind, Water, Forecast
data = np.array([
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes']
])

concepts = data[:, :-1]
target = data[:, -1]

s_final, g_final = learn(concepts, target)

print(f"Final Specific Boundary (S): {s_final}")
print(f"Final General Boundary (G): {g_final}")

Final Specific Boundary (S): ['Sunny' 'Warm' '?' 'Strong' '?' '?']
Final General Boundary (G): [[np.str_('Sunny'), '?', '?', '?', '?', '?'], ['?', np.str_('Warm'), '?', '?', '?', '?']]


In [4]:
def candidate_elimination(examples):
    n = len(examples[0]) - 1
    S = [['phi'] * n]
    G = [['?'] * n]

    for example in examples:
        # Separate the attributes from the label (Yes/No)
        attrs = example[:-1]
        label = example[-1]

        if label == 'Yes':
            # ---------------------------------------------------------
            # 1. Update the General boundary (G) for a POSITIVE example
            # Keep only the hypotheses in G that match this example.
            # ---------------------------------------------------------
            new_G = []
            for g in G:
                matches_example = True
                for i in range(n):
                    # If it's not a wildcard ('?') and doesn't match the attribute exactly, it fails
                    if g[i] != '?' and g[i] != attrs[i]:
                        matches_example = False
                        break
                
                if matches_example:
                    new_G.append(g)
            
            G = new_G

            # ---------------------------------------------------------
            # 2. Update the Specific boundary (S) for a POSITIVE example
            # Generalize S just enough to cover this new positive example.
            # ---------------------------------------------------------
            new_S = []
            for s in S:
                new_s = []
                for i in range(n):
                    if s[i] == 'phi':
                        # First time seeing a value, take the attribute
                        new_s.append(attrs[i])
                    elif s[i] != attrs[i]:
                        # Attribute conflicts with what we have, generalize to '?'
                        new_s.append('?')
                    else:
                        # Attribute matches what we already have, keep it
                        new_s.append(s[i])
                new_S.append(new_s)
                
            S = new_S

        else: # If label == 'No'
            # ---------------------------------------------------------
            # 3. Update the Specific boundary (S) for a NEGATIVE example
            # Remove any hypothesis in S that matches this negative example.
            # ---------------------------------------------------------
            new_S = []
            for s in S:
                is_consistent = False
                for i in range(n):
                    # A hypothesis is consistent if it differs from the negative example on at least one attribute
                    if s[i] != '?' and s[i] != attrs[i]:
                        is_consistent = True
                        break
                
                if is_consistent:
                    new_S.append(s)
            
            S = new_S

    return S, G

# Dataset
data = [
    ['Sunny','Warm','Normal','Strong','Warm','Same','Yes'],
    ['Sunny','Warm','High','Strong','Warm','Same','Yes'],
    ['Rainy','Cold','High','Strong','Warm','Change','No'],
    ['Sunny','Warm','High','Strong','Cool','Change','Yes'],
]

# Run the algorithm
S, G = candidate_elimination(data)

print("S (Specific boundary):", S)
print("G (General boundary) :", G)

S (Specific boundary): [['Sunny', 'Warm', '?', 'Strong', '?', '?']]
G (General boundary) : [['?', '?', '?', '?', '?', '?']]
